In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [2]:
df = pd.read_csv("fake_real_news.csv")

df.head()

,title,text,subject,date,target
0,GERMAN RESIDENTS FIGHT BACK: Anti-Islamic Song...,Apparently these Germans are not interested in...,left-news,"Jan 3, 2016",1
1,(VIDEO) BRAVO! TV HOST SCORCHES OBAMA FOR HIS ...,I VE HAD IT!,politics,"Jul 20, 2015",1
2,Greek president tells Turkey's Erdogan no trea...,ATHENS (Reuters) - Greek President Prokopis Pa...,worldnews,"December 7, 2017",0
3,Colbert Scorches Trump’s Anti-Trans Bigotry; ...,"During his campaign, Donald Trump promised tha...",News,"February 24, 2017",1
4,"Pentagon chief, Saudi deputy crown prince disc...",WASHINGTON (Reuters) - U.S. Defense Secretary ...,politicsNews,"March 16, 2017",0


In [3]:
df["content"] = df["title"] + " " + df["text"]

X = df["content"]
y = df["target"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [5]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [6]:
max_length = 200

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length)

In [ ]:
model = Sequential()

model.add(Embedding(input_dim=5000, output_dim=64, input_length=max_length))

model.add(LSTM(64))

model.add(Dense(1, activation="sigmoid"))

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [8]:
model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 45s 48ms/step - accuracy: 0.9624 - loss: 0.1100 - val_accuracy: 0.9868 - val_loss: 0.0450
Epoch 2/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 45s 50ms/step - accuracy: 0.9871 - loss: 0.0426 - val_accuracy: 0.9872 - val_loss: 0.0425
Epoch 3/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 46s 51ms/step - accuracy: 0.9908 - loss: 0.0339 - val_accuracy: 0.9890 - val_loss: 0.0423
Epoch 4/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 44s 48ms/step - accuracy: 0.9943 - loss: 0.0208 - val_accuracy: 0.9845 - val_loss: 0.0549
Epoch 5/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 43s 48ms/step - accuracy: 0.9895 - loss: 0.0354 - val_accuracy: 0.9833 - val_loss: 0.0599


In [9]:
pred = model.predict(X_test_pad)

pred = (pred > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

281/281 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step
Accuracy: 0.9800668151447661
              precision    recall  f1-score   support

           0       0.96      0.99      0.98      4241
           1       0.99      0.97      0.98      4739

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980

